# PromptTemplate in langchain and LCEL

In [6]:
# Suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

# OpenAI + LangChain imports
from langchain_openai import ChatOpenAI
from openai import OpenAI

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence, RunnableLambda
from langchain_core.messages import HumanMessage, SystemMessage

from langchain.chains import LLMChain  # still supported - It's deprecated but still works

from IPython.display import Markdown, display


In [7]:
params = {
    "max_tokens": 256,
    "temperature": 0.5, # Higher temperature increases the probability of selecting lower-probability tokens. - In my opinion, only use this to control diversity of outputs and not top_p
    "top_p": 0.2, # to control the randomness and diversity of the generated text - It is generally recommended to adjust one or the other, but not both simultaneously,
    "timeout":None,
}

llm = ChatOpenAI(
    model="gpt-4.1-nano",
    **params
)

template = """Tell me a {adjective} joke about {content}.
"""
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['adjective', 'content'], template='Tell me a {adjective} joke about {content}.\n')

In [8]:
prompt.format(adjective="funny", content="chickens")

'Tell me a funny joke about chickens.\n'

In [9]:
# Define a function to ensure proper formatting
def format_prompt(variables):
    return prompt.format(**variables)

Build a chain using the **LCEL (LangChain Expression Language)** pattern
- LangChain Expression Language (LCEL) is a pattern for building LangChain applications using the pipe operator (|) for more flexible composition. 
- It offers better composability, clearer visualization of data flow, and more flexibility when constructing complex chains.

In [13]:
# Create the chain with explicit formatting
joke_chain = (
    RunnableLambda(format_prompt)
    | llm 
)

# Run the chain
response = joke_chain.invoke({"adjective": "funny", "content": "chickens"})
print(response)

content="Sure! Here's a funny chicken joke for you:\n\nWhy did the chicken go to the séance?  \nTo get to the other side!" response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 15, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'finish_reason': 'stop', 'logprobs': None} id='run-fef645b3-b106-477c-9623-9053f01144b0-0' usage_metadata={'input_tokens': 15, 'output_tokens': 29, 'total_tokens': 44}


In [ ]:
# Alternate way to create a PromptTemplate without a format_prompt function
prompt_without_RunnableLambda = PromptTemplate(template=template, input_variables=["adjective", "content"], 
                                        #   partial_variables={"adjective": "funny"} # if you want to set a default value
                                          )

In [ ]:
# Create the chain with explicit formatting
joke_chain = (
    prompt_without_RunnableLambda
    | llm 
)

# Run the chain
response = joke_chain.invoke({"adjective": "funny", "content": "chickens"})
print(response)

In [ ]:
# Create the chain with explicit formatting
joke_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser() # extracts string outputs from LLM responses
)

# Run the chain
response = joke_chain.invoke({"adjective": "funny", "content": "chickens"})
print(response)

Sure! Here's a funny chicken joke for you:

Why did the chicken go to the séance?  
To get to the other side!


In [ ]:
content = """
    The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation. 
    Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems. 
    For instance, AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are making cities more efficient and reducing traffic congestion. 
    Moreover, online learning platforms are making education more accessible to people around the world, breaking down geographical and financial barriers. 
    These technological developments are not only enhancing productivity but also contributing to a more interconnected and informed society.
"""

template = """Summarize the content in one sentence.
            Here is content: {content}
"""
prompt = PromptTemplate.from_template(template) # redefine prompt. as its prompt is a global variable. It will update value in format_prompt function too.

# Create the LCEL chain
summarize_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
summary = summarize_chain.invoke({"content": content})
print(summary)

The rapid technological advancements in AI, machine learning, and the Internet of Things are transforming industries like healthcare, education, and transportation by improving efficiency, accessibility, and societal interconnectedness.


In [ ]:
role = """
    Chess game master
"""

tone = "engaging and immersive"

template = """
    You are an expert {role}. I have this question {question}. I would like our conversation to be {tone}.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
roleplay_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Create an interactive chat loop
while True:
    query = input("Question: ")
    
    if query.lower() in ["quit", "exit", "bye"]:
        print("Answer: Goodbye!")
        break
        
    response = roleplay_chain.invoke({"role": role, "question": query, "tone": tone})
    print("Answer: ", response)

# first question: How many plyeers can play chess?
# second question: quit

Answer:  Great question! Chess is traditionally a two-player game, where each player takes on the role of a king and queen, strategizing to checkmate their opponent. This classic format is what most people think of when they hear "chess."

However, the world of chess is much more diverse and creative! There are variants like **chess960** (Fischer Random), **bughouse**, and **team chess** that involve multiple players. For example:

- **Four-player chess**: Played on a special board with four sets of pieces, where two teams of two players compete against each other.
- **Team chess**: Multiple players form teams, collaborating to outsmart the opposing team.
- **Online multiplayer chess platforms**: Some allow more than two players to participate in a single game, either in free-for-alls or tournament formats.

In theory, there's no strict limit—if you have enough players and a suitable setup, you could have a giant game with dozens or even hundreds of participants! But practically, the m

In [ ]:
# Create the prompt template
template = """
Analyze the following product review:
"{review}"

Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)
"""

product_review_prompt = PromptTemplate.from_template(template)

# Create a formatting function
def format_review_prompt(variables):
    return product_review_prompt.format(**variables)

# Build the LCEL chain
review_analysis_chain = (
    RunnableLambda(format_review_prompt)
    | llm 
    | StrOutputParser()
)

# Process the reviews
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

for i, review in enumerate(reviews):
    print(f"==== Review #{i+1} ====")
    result = review_analysis_chain.invoke({"review": review})
    print(result)
    print()

==== Review #1 ====
- Sentiment: Positive
- Key Features Mentioned: Camera quality, battery life, heating issue during gaming
- Summary: The reviewer is highly satisfied with the smartphone's camera and battery performance but notes a minor heating problem during gaming.

==== Review #2 ====
- Sentiment: Negative
- Key Features Mentioned: Performance speed, stability (crashes), keyboard functionality, customer service
- Summary: The reviewer is highly dissatisfied with the laptop's performance, durability, and support.

